# 100+ Years Old Weather Stations Map
This notebook identifies all weather stations with at least 100 years of data and plots them on an interactive map using their latitude and longitude coordinates.

In [1]:
import pyarrow.dataset as ds
import pandas as pd
import folium

# Load dataset including Latitude and Longitude
dataset = ds.dataset('../data/dwd_historical_lake', format='parquet', partitioning='hive')
df = dataset.to_table(columns=['Station_ID', 'Date', 'Latitude', 'Longitude']).to_pandas()

# Calculate history length and keep the first coordinate for each station
summary = df.groupby('Station_ID').agg(
    Days_of_History=('Date', 'count'),
    Latitude=('Latitude', 'first'),
    Longitude=('Longitude', 'first')
).reset_index()

summary['Years_of_History'] = (summary['Days_of_History'] / 365.25).round(1)

# Filter for stations >= 100 years
stations_100 = summary[summary['Years_of_History'] >= 100].dropna(subset=['Latitude', 'Longitude'])

print(f"Found {len(stations_100)} stations with >= 100 years of history.")

import io
import requests

print("Fetching station names...")
url = 'https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt'
response = requests.get(url)
response.encoding = 'latin-1'
col_specs = [(0, 5), (6, 14), (15, 23), (24, 38), (39, 50), (51, 60), (61, 102), (102, 140)]
col_names = ["Station_ID", "Start_Date", "End_Date", "Altitude", "Latitude", "Longitude", "Station_Name", "Bundesland"]
df_meta_names = pd.read_fwf(io.StringIO(response.text), skiprows=2, names=col_names, colspecs=col_specs, dtype={"Station_ID": str})
df_meta_names['Station_ID'] = df_meta_names['Station_ID'].str.zfill(5)
df_meta_names['Station_Name'] = df_meta_names['Station_Name'].str.strip()
name_dict = df_meta_names.set_index('Station_ID')['Station_Name'].to_dict()


Found 47 stations with >= 100 years of history.
Fetching station names...


In [2]:
# Create an interactive map centered on Germany
m = folium.Map(location=[51.1657, 10.4515], zoom_start=6)

for idx, row in stations_100.iterrows():
    name = name_dict.get(row['Station_ID'], 'Unknown')
    folium.Marker(
        [row['Latitude'], row['Longitude']],
        popup=f"Station: {name} ({row['Station_ID']})<br>History: {row['Years_of_History']} years"
    ).add_to(m)

# Display the map
m